
# CRISP-DM Step 5: **Evaluation** — Hands-on Lab

**วัตถุประสงค์**
- ประเมินว่าโมเดล **ตอบโจทย์ทางธุรกิจ** จริงหรือไม่ (เชื่อมเมตริกกับต้นทุน/ผลประโยชน์)
- วัดผลด้วยเมตริกที่เหมาะสม (Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC)
- ทดลอง **ปรับ Threshold** เพื่อ Optimize ตามเป้าหมายธุรกิจ
- คำนวณ **Cost–Benefit** และทำ **Bootstrap CI** ของเมตริก
- สรุปเป็น **One-page Model Report**



## 0) ตั้งค่า Business Scenario (แก้ไขได้)
- `PROFIT_PER_TP` = กำไรสุทธิต่อ 1 เคสที่ทำนาย "จะซื้อ" และเกิดการซื้อจริง
- `COST_PER_CONTACT` = ต้นทุนการติดต่อ/เคส ที่โมเดลทำนายว่า "จะซื้อ"
> *หมายเหตุ:* ตัวอย่างนี้ใช้โจทย์ "คัดรายชื่อลูกค้าที่น่าจะซื้อ" (binary classification)


In [ ]:

PROFIT_PER_TP = 300.0   # บาท
COST_PER_CONTACT = 20.0 # บาท



## 1) สร้างชุดข้อมูลสังเคราะห์ (Self-contained)
ฟีเจอร์หลัก: `recency_days, frequency, monetary, avg_orders_per_month, avg_basket_value, product_pref`  
เป้าหมาย: `will_buy` (0/1)


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

rng = np.random.default_rng(2025)
n = 6000

recency_days = rng.integers(0, 200, size=n)
frequency = rng.poisson(5, size=n) + 1
monetary = rng.gamma(shape=2.0, scale=120.0, size=n)
avg_orders_per_month = np.clip(rng.normal(2.5, 1.2, size=n), 0, None)
avg_basket_value = monetary / np.maximum(frequency, 1)

categories = np.array(["Electronics","Grocery","Fashion","Beauty","Home","Sports"])
product_pref = rng.choice(categories, size=n, replace=True)

# สร้าง y แบบมีสัญญาณจากฟีเจอร์ + noise
z = 0.8*(frequency>5) + 0.6*(avg_orders_per_month>2.5) - 0.01*recency_days + 0.001*monetary + rng.normal(0, 0.5, size=n)
prob = 1/(1+np.exp(-z))
will_buy = (rng.random(n) < prob).astype(int)

feat = pd.DataFrame({
    "recency_days": recency_days,
    "frequency": frequency,
    "monetary": monetary.round(2),
    "avg_orders_per_month": avg_orders_per_month.round(2),
    "avg_basket_value": avg_basket_value.round(2),
    "product_pref": product_pref,
    "will_buy": will_buy
})

display(feat.head())
print("Shape:", feat.shape, "| Positive rate:", feat['will_buy'].mean().round(3))



## 2) Train/Test + Pipeline (Logistic Regression)
ใช้ Pipeline เพื่อรวมขั้นตอนเตรียมข้อมูลและโมเดล ลดโอกาสเกิด **data leakage**


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay

feature_cols_num = ["recency_days","frequency","monetary","avg_orders_per_month","avg_basket_value"]
feature_cols_cat = ["product_pref"]
X = feat[feature_cols_num + feature_cols_cat].copy()
y = feat["will_buy"].astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), feature_cols_num),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), feature_cols_cat)
])

clf = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000))
])

clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)
y_prob = clf.predict_proba(X_te)[:,1]

print(classification_report(y_te, y_pred, digits=3))
print("ROC-AUC:", roc_auc_score(y_te, y_prob))



### ROC Curve


In [ ]:

RocCurveDisplay.from_predictions(y_te, y_prob)



### Precision–Recall Curve


In [ ]:

PrecisionRecallDisplay.from_predictions(y_te, y_prob)



## 3) Threshold Tuning + **Cost–Benefit**

เราจะสแกน threshold เพื่อหา **Net Benefit = TP×PROFIT_PER_TP − (Predicted Positives)×COST_PER_CONTACT**


In [ ]:

from sklearn.metrics import precision_recall_fscore_support

def sweep_thresholds(y_true, y_proba, profit_per_tp, cost_per_contact, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 19)
    rows = []
    for th in thresholds:
        y_hat = (y_proba >= th).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(y_true, y_hat, average="binary", zero_division=0)
        tp = int(((y_true==1) & (y_hat==1)).sum())
        pp = int(y_hat.sum())
        net = tp*profit_per_tp - pp*cost_per_contact
        rows.append((th, pp, tp, p, r, f1, net))
    return pd.DataFrame(rows, columns=["threshold","pred_pos","TP","precision","recall","f1","net_benefit"])

th_table = sweep_thresholds(y_te, y_prob, PROFIT_PER_TP, COST_PER_CONTACT)
th_table = th_table.round({"precision":3, "recall":3, "f1":3})
display(th_table)



### F1 vs Threshold


In [ ]:

plt.figure()
plt.plot(th_table["threshold"], th_table["f1"], marker="o")
plt.title("F1 vs Threshold")
plt.xlabel("Threshold")
plt.ylabel("F1")
plt.show()



### Net Benefit vs Threshold


In [ ]:

plt.figure()
plt.plot(th_table["threshold"], th_table["net_benefit"], marker="o")
plt.title("Net Benefit vs Threshold")
plt.xlabel("Threshold")
plt.ylabel("Net Benefit (Baht)")
plt.show()



### เลือก Threshold ที่ให้ **Net Benefit สูงสุด**


In [ ]:

best_row = th_table.iloc[th_table["net_benefit"].idxmax()]
best_th = float(best_row["threshold"])
print("Best threshold by Net Benefit:", best_th)
print(best_row)

y_hat_best = (y_prob >= best_th).astype(int)
ConfusionMatrixDisplay.from_predictions(y_te, y_hat_best)



## 4) Bootstrap 95% CI ของ ROC-AUC และ F1 (ที่ Threshold ที่เลือก)


In [ ]:

from sklearn.utils import resample
from sklearn.metrics import f1_score

def bootstrap_ci_metric(y_true, y_proba, metric_fn, n_boot=400, random_state=123):
    rng = np.random.default_rng(random_state)
    vals = []
    idx = np.arange(len(y_true))
    for _ in range(n_boot):
        b = rng.choice(idx, size=len(idx), replace=True)
        vals.append(metric_fn(y_true[b], y_proba[b]))
    vals = np.array(vals)
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return vals.mean(), lo, hi

# ROC-AUC (threshold-independent)
from sklearn.metrics import roc_auc_score
auc_mean, auc_lo, auc_hi = bootstrap_ci_metric(y_te.values, y_prob, roc_auc_score, n_boot=400)
print(f"ROC-AUC bootstrap mean={auc_mean:.3f}, 95% CI=({auc_lo:.3f}, {auc_hi:.3f})")

# F1 at best threshold
def f1_at_th(y_true, y_proba, th):
    y_hat = (y_proba >= th).astype(int)
    return f1_score(y_true, y_hat)

f1_mean, f1_lo, f1_hi = bootstrap_ci_metric(y_te.values, y_prob, lambda yt, yp: f1_at_th(yt, yp, best_th), n_boot=400)
print(f"F1@{best_th:.2f} bootstrap mean={f1_mean:.3f}, 95% CI=({f1_lo:.3f}, {f1_hi:.3f})")



## 5) **One-page Model Report** (Template)
แก้ไขค่าด้านล่างเพื่อสร้างสรุปรายงาน 1 หน้า


In [ ]:

REPORT = {
    "business_goal": "เพิ่มยอดขายผ่านการติดต่อเชิงรุก",
    "model_name": "Logistic Regression (v1.0)",
    "data_scope": "ลูกค้า 6,000 ราย (split train/test 80/20)",
    "metrics_offline": {
        "roc_auc": float(roc_auc_score(y_te, y_prob))
    },
    "selected_threshold": float(best_th),
    "profit_per_tp": float(PROFIT_PER_TP),
    "cost_per_contact": float(COST_PER_CONTACT),
}

summary = f'''
# Model Report

**Business Goal:** {REPORT["business_goal"]}
**Model:** {REPORT["model_name"]}
**Data:** {REPORT["data_scope"]}

**Offline Metrics (Test):**
- ROC-AUC: {REPORT["metrics_offline"]["roc_auc"]:.3f}
- F1@{REPORT["selected_threshold"]:.2f}: {f1_at_th(y_te.values, y_prob, REPORT["selected_threshold"]):.3f}

**Business Assumptions:**
- Profit/TP = {REPORT["profit_per_tp"]:.0f} THB
- Cost/Contact = {REPORT["cost_per_contact"]:.0f} THB

**Chosen Threshold:** {REPORT["selected_threshold"]:.2f}
- Expected Net Benefit (at chosen th): {float(th_table.loc[th_table["threshold"]==REPORT["selected_threshold"], "net_benefit"].values[0]):.0f} THB

**Notes / Risks:**
- พิจารณา A/B Test 2–4 สัปดาห์ ติดตาม Net Revenue และ Conversion จริง
- เฝ้าระวัง Drift ตามฤดูกาล/แคมเปญ
- ทบทวน Threshold รายสัปดาห์
'''
print(summary)
